# Detecção de Objetos com YOLOv8n na Webcam do Google Colab
Este notebook demonstra como usar o modelo YOLOv8n para detecção de objetos em uma imagem capturada a partir da sua webcam. O código adapta a lógica de captura do Darknet para a biblioteca `ultralytics`, mais moderna e eficiente.

## 1. Instalação e Configuração do YOLOv8
Primeiro, instalamos a biblioteca `ultralytics` e importamos as dependências necessárias. Em seguida, carregamos o modelo YOLOv8n pré-treinado.

In [ ]:
# Instala o pacote ultralytics (se já estiver instalado, ele apenas atualizará)
!pip install ultralytics

# Importa as bibliotecas necessárias
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode, b64encode
import PIL
import io
import html
import time

# Carrega o modelo YOLOv8n pré-treinado
yolo_model = YOLO('yolov8n.pt')

print("Modelo YOLOv8n carregado com sucesso!")

## 2. Funções Auxiliares de Captura
Essas funções, adaptadas do notebook original, permitem a captura de uma imagem estática da sua webcam no ambiente do Colab.

In [ ]:
# Função para converter o objeto JavaScript em uma imagem OpenCV
def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  img = js_to_image(data)
  cv2.imwrite(filename, img)
  return filename


## 3. Capturar Imagem e Executar a Detecção
Clique no botão 'Capture' para tirar uma foto. Em seguida, o script passará a imagem para o modelo YOLOv8n, que detectará os objetos, e a imagem com as caixas delimitadoras será exibida como resultado.

In [ ]:
# Captura a foto da webcam
try:
    filename = take_photo('webcam_photo.jpg')
    print(f'Foto salva como: {filename}')

    # Carrega a imagem capturada
    image = cv2.imread(filename)
    
    # Executa a detecção de objetos com o YOLOv8n
    results = yolo_model(source=image, save=True, conf=0.25)

    # A biblioteca ultralytics já salva a imagem anotada em 'runs/detect/...'
    # Vamos carregar e exibir essa imagem anotada.
    output_path = results[0].save_dir + '/' + results[0].path.split('/')[-1]
    
    print(f'Imagem com detecções salva em: {output_path}')
    
    # Exibe a imagem anotada no notebook
    display(Image(filename=output_path))

except Exception as err:
    print(str(err))